# Reinforcement Loop — Simple Working Examples
## Two approaches: LangChain + LLM  |  CrewAI

**What is a Reinforcement Loop?**
```
ACT    → Agent takes an action / gives a response
OBSERVE → We score the result (feedback)
ADAPT  → If score is low, agent rewrites its own strategy
```
No complex RL algorithms. Just: act → score → improve.


## Setup

In [ ]:
# pip install langchain-openai crewai==0.28.8 python-dotenv

import os, warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("Ready ✓")


---
# PART 1 — LangChain Reinforcement Loop
**Scenario:** A customer support agent learns to give better responses over 3 iterations.


In [ ]:
# ── PART 1: LangChain Reinforcement Loop ──────────────────
from langchain.schema import HumanMessage, SystemMessage

# The agent's current strategy (starts basic, improves each loop)
strategy = "You are a helpful customer support agent. Answer the customer's question."

# Memory of what worked / didn't
learning_history = []

def act(customer_message: str) -> str:
    '''Agent responds using its current strategy.'''
    messages = [
        SystemMessage(content=strategy),
        HumanMessage(content=customer_message)
    ]
    response = llm.invoke(messages)
    return response.content

def observe(response: str, feedback_score: int) -> None:
    '''Log the result. Score 1-10.'''
    learning_history.append({
        "response_snippet": response[:80],
        "score": feedback_score
    })
    print(f"  Score: {feedback_score}/10")

def adapt() -> None:
    '''If recent score is low, ask LLM to rewrite the strategy.'''
    global strategy
    if not learning_history:
        return
    recent_score = learning_history[-1]["score"]
    if recent_score < 7:
        prompt = f'''
Current strategy: "{strategy}"
Last response scored {recent_score}/10.
Rewrite the strategy in 1-2 sentences to get a better score next time.
Return ONLY the new strategy text.
'''
        new_strategy = llm.invoke([HumanMessage(content=prompt)]).content.strip()
        strategy = new_strategy
        print(f"  Strategy updated: {strategy[:80]}...")
    else:
        print(f"  Strategy kept (score was good).")

print("Functions defined ✓")


In [ ]:
# ── Run 3 iterations ──────────────────────────────────────
customer_query = "I was charged twice for my subscription. What do I do?"

# Simulated feedback scores (in real life: human rating or automated metric)
feedback_scores = [4, 6, 8]

for i, score in enumerate(feedback_scores):
    print(f"\n{'='*50}")
    print(f"ITERATION {i+1}")
    print(f"{'='*50}")

    # ACT
    print("\nACT — Agent responding...")
    response = act(customer_query)
    print(f"Response: {response[:120]}...")

    # OBSERVE
    print("\nOBSERVE — Scoring the response...")
    observe(response, score)

    # ADAPT
    print("\nADAPT — Reviewing strategy...")
    adapt()

print("\n✓ Reinforcement loop complete.")
print(f"\nFinal strategy:\n{strategy}")


In [ ]:
# ── View learning history ──────────────────────────────────
print("Learning History:")
for i, entry in enumerate(learning_history):
    print(f"  Iteration {i+1}: score={entry['score']} | {entry['response_snippet']}")


---
# PART 2 — CrewAI Reinforcement Loop
**Scenario:** A travel planning crew improves its output quality over 3 iterations.
The crew's risk strategy (backstory) updates based on quality score.


In [ ]:
# ── PART 2: CrewAI Reinforcement Loop ─────────────────────
from crewai import Agent, Task, Crew

# Starting strategy — will evolve each iteration
risk_strategy = "Provide a basic travel recommendation with budget and key activities."

# Learning history
crew_history = []

def build_crew(current_strategy: str) -> Crew:
    '''Rebuild crew with updated strategy injected into agent backstory.'''

    planner = Agent(
        role="Travel Planner",
        goal="Create a helpful travel recommendation for the user.",
        backstory=current_strategy,   # ← strategy injected here
        llm=llm,
        verbose=False,
        allow_delegation=False,
        max_iter=3
    )

    reviewer = Agent(
        role="Quality Reviewer",
        goal="Score the travel plan from 1-10 and explain what is missing.",
        backstory="You are a strict travel editor. Score plans on clarity, budget detail, and practicality.",
        llm=llm,
        verbose=False,
        allow_delegation=False,
        max_iter=3
    )

    plan_task = Task(
        description="Create a 3-day travel plan for a family visiting Rome. Include budget, hotels, and activities.",
        expected_output="A structured 3-day plan with budget breakdown.",
        agent=planner
    )

    review_task = Task(
        description="Review the travel plan above. Score it 1-10. State what is missing. Output: score and feedback.",
        expected_output="Score (1-10) and 2-3 bullet points of feedback.",
        agent=reviewer
    )

    return Crew(
        agents=[planner, reviewer],
        tasks=[plan_task, review_task],
        verbose=False
    )

def extract_score(crew_output: str) -> int:
    '''Pull numeric score from reviewer output. Default 5 if not found.'''
    import re
    matches = re.findall(r'([1-9]|10)', str(crew_output))
    return int(matches[0]) if matches else 5

def update_strategy(current: str, score: int, feedback: str) -> str:
    '''Ask LLM to improve the strategy if score < 7.'''
    if score >= 7:
        return current
    prompt = f'''
Current planner strategy: "{current}"
Quality score: {score}/10
Reviewer feedback: "{feedback[:200]}"
Rewrite the strategy in 1-2 sentences so the planner scores higher next time.
Return ONLY the new strategy text.
'''
    return llm.invoke([HumanMessage(content=prompt)]).content.strip()

print("CrewAI functions defined ✓")


In [ ]:
# ── Run 3 crew iterations ─────────────────────────────────
from langchain.schema import HumanMessage

for iteration in range(1, 4):
    print(f"\n{'='*50}")
    print(f"ITERATION {iteration}")
    print(f"{'='*50}")
    print(f"Strategy: {risk_strategy[:80]}...")

    # ACT — run the crew
    crew = build_crew(risk_strategy)
    result = crew.kickoff()
    output_text = str(result)

    # OBSERVE — extract score
    score = extract_score(output_text)
    print(f"\nQuality Score: {score}/10")
    print(f"Output snippet: {output_text[:150]}...")

    crew_history.append({"iteration": iteration, "score": score,
                          "strategy": risk_strategy[:80]})

    # ADAPT — update strategy if needed
    risk_strategy = update_strategy(risk_strategy, score, output_text)
    if score < 7:
        print(f"Strategy updated for next iteration.")
    else:
        print(f"Strategy kept — score is acceptable.")

print("\n✓ CrewAI reinforcement loop complete.")


In [ ]:
# ── Summary ───────────────────────────────────────────────
print("Iteration Summary:")
print(f"{'Iter':<6} {'Score':<8} {'Strategy Snippet'}")
print("-" * 60)
for entry in crew_history:
    print(f"{entry['iteration']:<6} {entry['score']:<8} {entry['strategy']}")

print(f"\nFinal strategy:\n{risk_strategy}")


---
## Key Takeaway

| | LangChain | CrewAI |
|---|---|---|
| Agent | `SystemMessage` + `HumanMessage` | `Agent(backstory=strategy)` |
| Act | `llm.invoke(messages)` | `crew.kickoff()` |
| Observe | Manual score | Reviewer agent scores |
| Adapt | LLM rewrites `strategy` string | LLM rewrites `risk_strategy` → injected into next crew |

**The loop is the same in both. Only the framework changes.**
